Library Check

In [2]:
# Standard library
import json
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path

# Data
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# XML parsing
from lxml import etree

# Stats and plots
from scipy.stats import spearmanr, kendalltau
import matplotlib.pyplot as plt

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
plt.rcParams["figure.figsize"] = (10, 5)

XML file Check

In [6]:
# Checking file existence
XML_PATH = Path(r"C:\Users\Mark\Documents\ITB\DataMining\TugasKelompok\dblp-2025-12-01.xml")
print(XML_PATH.exists(), XML_PATH.stat().st_size / 1e9, "GB")

True 4.939110596 GB


XML Hash check

In [25]:
import hashlib

def sha256_of(path, block=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(block):
            h.update(chunk)
    return h.hexdigest()

print("File:", XML_PATH.name)
print("Size (bytes):", XML_PATH.stat().st_size)
print("SHA-256:", sha256_of(XML_PATH))

File: dblp-2025-12-01.xml
Size (bytes): 4939110596
SHA-256: 0f3dbcbd332674bd2e292395367888d1d39dec2a497d08d5c08b1d8298907d3e


Check XML DTD scheme

In [ ]:
with open(XML_PATH, "rb") as f:
    head = f.read(2000).decode("latin-1")

m = re.search(r'<!DOCTYPE\s+\w+\s+SYSTEM\s+"([^"]+)"', head)
DTD_PATH = XML_PATH.parent / m.group(1)

print("Expected DTD:", DTD_PATH.name)
print("DTD found:   ", DTD_PATH.exists())

Expected DTD: dblp-2023-06-28.dtd
DTD found:    True
<!--
	The dblp computer science bibliography is copyright by Schloss Dagstuhl - Leibniz Center for Informatics.

	The me


Extraction config and output schema

In [21]:
OUT_DIR = Path("data")
OUT_DIR.mkdir(exist_ok=True)

YEAR_MIN, YEAR_MAX = 2015, 2025
CHUNK_ROWS = 100_000        # rows buffered in RAM before writing to parquet
MAX_RECORDS = None     # quick test run; set to None for the full pass

SUFFIX = "_test" if MAX_RECORDS else ""
RAW_PARQUET = OUT_DIR / f"raw_inproceedings{SUFFIX}.parquet"
COUNTS_JSON = OUT_DIR / f"raw_counts{SUFFIX}.json"

REC_TAGS = {"article", "inproceedings", "proceedings", "book",
            "incollection", "phdthesis", "mastersthesis",
            "www", "person", "data"}

SCHEMA = pa.schema([
    ("paper_key",  pa.string()),
    ("authors",    pa.list_(pa.string())),
    ("title",      pa.string()),
    ("year_raw",   pa.string()),
    ("booktitle",  pa.string()),
    ("crossref",   pa.string()),
    ("key_prefix", pa.string()),
    ("venue_key",  pa.string()),
])

Helper to turn one inproceedings element into a row

In [22]:
def text_of(el):
    """All text inside an element (handles inline tags like <i>), or None."""
    return "".join(el.itertext()).strip() if el is not None else None

def parse_inproceedings(el):
    key = el.get("key")
    parts = key.split("/") if key else []
    return {
        "paper_key":  key,
        "authors":    [text_of(a) for a in el.findall("author")],
        "title":      text_of(el.find("title")),
        "year_raw":   text_of(el.find("year")),
        "booktitle":  text_of(el.find("booktitle")),
        "crossref":   text_of(el.find("crossref")),
        "key_prefix": parts[0] if parts else None,                   # e.g. conf
        "venue_key":  parts[1] if len(parts) >= 3 else None,         # e.g. kdd
    }

Streaming extraction pass (counts all types, saves 2015-2025 inproceedings)

In [23]:
def run_extraction():
    type_counts, year_counts, prefix_counts = Counter(), Counter(), Counter()
    total = kept = bad_year = 0
    buffer = []
    t0 = time.time()

    tmp = RAW_PARQUET.with_name(RAW_PARQUET.name + ".tmp")
    writer = pq.ParquetWriter(tmp, SCHEMA)

    def flush():
        nonlocal buffer
        if buffer:
            writer.write_table(pa.Table.from_pylist(buffer, schema=SCHEMA))
            buffer = []

    try:
        ctx = etree.iterparse(str(XML_PATH), events=("end",),
                              load_dtd=True, resolve_entities=True,
                              huge_tree=True)
        for _, el in ctx:
            if el.tag not in REC_TAGS:
                continue
            total += 1
            type_counts[el.tag] += 1

            if el.tag == "inproceedings":
                rec = parse_inproceedings(el)
                prefix_counts[str(rec["key_prefix"])] += 1
                y = rec["year_raw"]
                if y and y.isdigit():
                    year_counts[y] += 1                  # all years, pre-filter
                    if YEAR_MIN <= int(y) <= YEAR_MAX:
                        buffer.append(rec)
                        kept += 1
                        if len(buffer) >= CHUNK_ROWS:
                            flush()
                else:
                    bad_year += 1

            el.clear()
            while el.getprevious() is not None:
                del el.getparent()[0]

            if total % 500_000 == 0:
                print(f"{total:,} records, {kept:,} kept, {time.time()-t0:.0f}s")
            if MAX_RECORDS and total >= MAX_RECORDS:
                break
        flush()
    finally:
        writer.close()

    tmp.replace(RAW_PARQUET)   # only a finished file gets the final name

    meta = {
        "snapshot": "dblp-2025-12-01",
        "dtd": DTD_PATH.name,
        "xml_size_bytes": XML_PATH.stat().st_size,
        "year_filter": [YEAR_MIN, YEAR_MAX],
        "extracted_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "elapsed_seconds": round(time.time() - t0, 1),
        "total_records": total,
        "record_type_counts": dict(type_counts),
        "inproceedings_all_years_by_year": dict(sorted(year_counts.items())),
        "inproceedings_by_key_prefix": dict(prefix_counts),
        "inproceedings_missing_or_invalid_year": bad_year,
        "rows_kept": kept,
    }
    COUNTS_JSON.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"Done: {total:,} records, {kept:,} rows kept, {meta['elapsed_seconds']}s")

if RAW_PARQUET.exists() and COUNTS_JSON.exists():
    print("Already extracted, skipping:", RAW_PARQUET)
else:
    run_extraction()

500,000 records, 215,005 kept, 32s
1,000,000 records, 471,693 kept, 66s
1,500,000 records, 724,653 kept, 100s
2,000,000 records, 957,848 kept, 133s
2,500,000 records, 1,204,584 kept, 168s
3,000,000 records, 1,443,194 kept, 202s
3,500,000 records, 1,699,733 kept, 236s
4,000,000 records, 1,898,822 kept, 267s
4,500,000 records, 1,898,822 kept, 271s
5,000,000 records, 1,898,822 kept, 275s
5,500,000 records, 1,898,822 kept, 280s
6,000,000 records, 1,898,822 kept, 284s
6,500,000 records, 1,898,822 kept, 288s
7,000,000 records, 1,898,822 kept, 293s
7,500,000 records, 1,898,822 kept, 297s
8,000,000 records, 1,899,209 kept, 305s
8,500,000 records, 1,899,209 kept, 318s
9,000,000 records, 1,899,209 kept, 331s
9,500,000 records, 1,899,209 kept, 343s
10,000,000 records, 1,899,209 kept, 356s
10,500,000 records, 1,899,209 kept, 368s
11,000,000 records, 1,899,209 kept, 379s
11,500,000 records, 1,900,342 kept, 391s
12,000,000 records, 1,901,774 kept, 402s
Done: 12,169,828 records, 1,901,774 rows kept, 

### Task 1 — Data Understanding and Preprocessing 

##### 1.1 Dataset overview and consistency checks

In [27]:
from IPython.display import display

raw = pd.read_parquet(RAW_PARQUET)
meta = json.loads(COUNTS_JSON.read_text(encoding="utf-8"))

# --- A. Dataset-level facts ---
print("=== A. Dataset ===")
print("Snapshot:", meta["snapshot"], "| DTD:", meta["dtd"])
print("XML size (bytes):", f"{meta['xml_size_bytes']:,}")
print("Total records:", f"{meta['total_records']:,}")
print("Extract used for this project:", raw.shape, "(rows, columns)")

# --- B. Structure of inproceedings, from the DTD ---
print("\n=== B. inproceedings structure (DTD) ===")
dtd = DTD_PATH.read_text(encoding="latin-1")
for pat in [r'<!ENTITY\s+%\s+field\s+"[^"]*"\s*>',
            r'<!ELEMENT\s+inproceedings\b[^>]*>',
            r'<!ATTLIST\s+inproceedings\b[^>]*>']:
    m = re.search(pat, dtd, re.S)
    print(m.group(0) if m else f"(not found: {pat})", "\n")

# --- C. Publication types ---
print("=== C. Record types ===")
tbl = (pd.Series(meta["record_type_counts"])
         .sort_values(ascending=False).to_frame("jumlah_record"))
tbl["persen"] = (tbl["jumlah_record"] / meta["total_records"] * 100).round(2)
tbl.loc["TOTAL"] = [meta["total_records"], 100.0]
tbl["jumlah_record"] = tbl["jumlah_record"].astype("int64")
display(tbl)

# --- D. Attributes of the extract ---
print("\n=== D. Extract sample ===")
display(raw.head())

# --- E. Consistency checks ---
print("\n=== E. Checks ===")
yrs = meta["inproceedings_all_years_by_year"]
n_in = meta["record_type_counts"]["inproceedings"]
in_range = {y: v for y, v in yrs.items() if YEAR_MIN <= int(y) <= YEAR_MAX}

checks = {
    "record types sum to total_records":
        sum(meta["record_type_counts"].values()) == meta["total_records"],
    "key prefixes sum to inproceedings":
        sum(meta["inproceedings_by_key_prefix"].values()) == n_in,
    "all-years counts + invalid years = inproceedings":
        sum(yrs.values()) + meta["inproceedings_missing_or_invalid_year"] == n_in,
    "in-range years sum to rows_kept":
        sum(in_range.values()) == meta["rows_kept"],
    "parquet rows == rows_kept":
        len(raw) == meta["rows_kept"],
    "parquet year counts == JSON year counts":
        raw["year_raw"].value_counts().to_dict() == in_range,
    "paper_key unique":
        raw["paper_key"].is_unique,
    "no nulls in paper_key / year_raw":
        bool(raw[["paper_key", "year_raw"]].notna().all().all()),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)

print("\nExtraction time (s):", meta["elapsed_seconds"])
print("Invalid/missing year (inproceedings, all years):",
      meta["inproceedings_missing_or_invalid_year"])

=== A. Dataset ===
Snapshot: dblp-2025-12-01 | DTD: dblp-2023-06-28.dtd
XML size (bytes): 4,939,110,596
Total records: 12,169,828
Extract used for this project: (1901774, 8) (rows, columns)

=== B. inproceedings structure (DTD) ===
<!ENTITY % field "author|editor|title|booktitle|pages|year|address|journal|volume|number|month|url|ee|cdrom|cite|publisher|note|crossref|isbn|series|school|chapter|publnr|stream|rel"> 

<!ELEMENT inproceedings (%field;)*> 

<!ATTLIST inproceedings key CDATA #REQUIRED
                        mdate CDATA #IMPLIED
                        publtype NMTOKENS #IMPLIED
                        cdate CDATA #IMPLIED
> 

=== C. Record types ===


,jumlah_record,persen
article,4101061,33.70
www,3955829,32.51
inproceedings,3788716,31.13
phdthesis,149054,1.22
incollection,70988,0.58
proceedings,63072,0.52
book,21291,0.17
data,19790,0.16
mastersthesis,27,0.00
TOTAL,12169828,100.00



=== D. Extract sample ===


,paper_key,authors,title,year_raw,booktitle,crossref,key_prefix,venue_key
0,conf/dac/RoquetSRTSK24,"[Lucas Roquet, Fernando Fernandes dos Santos, ...",Cross-Layer Reliability Evaluation and Efficie...,2024,DAC,conf/dac/2024,conf,dac
1,conf/dac/LiuZWCYHYJ22,"[Fangxin Liu, Wenbo Zhao 0005, Zongwu Wang, Yo...",SATO: spiking neural network acceleration via ...,2022,DAC,conf/dac/2022,conf,dac
2,conf/dac/AhnYLHKP20,"[Sunwoo Ahn, Hayoon Yi, Younghan Lee, Whoi Ree...",Hawkware: Network Intrusion Detection based on...,2020,DAC,conf/dac/2020,conf,dac
3,conf/dac/ParkPVC24,"[Chanwoo Park, Junghwan Park, Premkumar Vincen...",Accelerating DTCO with a Sample-Efficient Acti...,2024,DAC,conf/dac/2024,conf,dac
4,conf/dac/PuL0HLC024,"[Yuan Pu 0001, Fangzhou Liu, Yu Zhang 0189, Zh...",Lesyn: Placement-aware Logic Resynthesis for N...,2024,DAC,conf/dac/2024,conf,dac



=== E. Checks ===
PASS - record types sum to total_records
PASS - key prefixes sum to inproceedings
PASS - all-years counts + invalid years = inproceedings
PASS - in-range years sum to rows_kept
PASS - parquet rows == rows_kept
PASS - parquet year counts == JSON year counts
PASS - paper_key unique
PASS - no nulls in paper_key / year_raw

Extraction time (s): 405.1
Invalid/missing year (inproceedings, all years): 0


**Sumber dataset**. Dataset yang digunakan adalah DBLP computer science bibliography dalam rilis Monthly Snapshot XML Release of December 2025 (dblp-2025-12-01.xml), yang didistribusikan oleh Schloss Dagstuhl – Leibniz-Zentrum für Informatik (DOI: 10.4230/dblp.xml.2025-12-01; https://drops.dagstuhl.de/entities/artifact/10.4230/dblp.xml.2025-12-01). Struktur berkas mengacu pada DTD rilis 28 Juni 2023 (dblp-2023-06-28.dtd). Snapshot bertanggal 1 Desember 2025 dan diunduh pada 16 November 2026. Berkas XML hasil dekompresi berukuran 4.939.110.596 byte (SHA-256: **0f3dbcbd332674bd2e292395367888d1d39dec2a497d08d5c08b1d8298907d3e**).

**Jumlah data**. Berkas mentah berisi 12.169.828 record dari sembilan tipe: *article* (4.101.061), *www* (3.955.829), *inproceedings* (3.788.716), *phdthesis* (149.054), *incollection* (70.988), *proceedings* (63.072), *book* (21.291), *data* (19.790), dan *mastersthesis* (27). Sesuai ketentuan proyek, hanya inproceedings dengan tahun 2015–2025 yang diekstrak, yaitu 1.901.774 record (50,2% dari seluruh *inproceedings*). Setelah preprocessing, jumlah publikasi yang digunakan dalam analisis adalah [N_final].

#### 1.2 Preprocessing

Setup : Load the raw extract and start the funnel log

In [28]:
df = pd.read_parquet(RAW_PARQUET)   # raw stays untouched on disk; we only change df

funnel = []
def log_step(step, before, after, note=""):
    funnel.append({"step": step, "rows_before": before, "rows_after": after,
                   "removed": before - after, "note": note})

log_step("Raw extract: inproceedings, 2015-2025", len(df), len(df))
display(pd.DataFrame(funnel))

,step,rows_before,rows_after,removed,note
0,"Raw extract: inproceedings, 2015-2025",1901774,1901774,0,


Step 1: select conference and workshop papers

Decision point: the default keeps only conf and drops the rest. I chose that because the task is about conference popularity, and a non-conf venue key would show up as a “conference” in your rankings. That reasoning has a weakness: I don’t know what the journals/ rows are, and some could be genuine conference papers. Read the examples this cell prints. If they look like real conference papers, add the prefix to KEEP_PREFIXES and rerun. Whichever you choose, put the rule and its reason in your methodology

In [29]:
# Step 1: select conference and workshop papers
KEEP_PREFIXES = {"conf"}      # decision point: see note below

print("Key prefixes before:")
print(df["key_prefix"].value_counts(), "\n")

mask = df["key_prefix"].isin(KEEP_PREFIXES)
print("Examples of rows that would be dropped:")
display(df.loc[~mask, ["paper_key", "booktitle"]].head(10))

before = len(df)
df = df.loc[mask].copy()
log_step("Keep conference/workshop keys", before, len(df),
         f"prefixes kept: {sorted(KEEP_PREFIXES)}")
display(pd.DataFrame(funnel))

Key prefixes before:
key_prefix
conf        1898822
journals       2912
books            40
Name: count, dtype: int64 

Examples of rows that would be dropped:


,paper_key,booktitle
1898822,journals/entcs/GudmundssonMS16,ICTCS
1898823,journals/entcs/SuLC17,ISDT
1898824,journals/entcs/Keimel15,MFPS
1898825,journals/entcs/WangRW16,PASM
1898826,journals/entcs/CiabattoniLOP17,LSFA
1898827,journals/entcs/Staton18,MFPS
1898828,journals/entcs/AlconGM19,LAGOS
1898829,journals/entcs/FourneauP16,PASM
1898830,journals/entcs/PrabhakarS15,NSV
1898831,journals/entcs/Weffort-SantosC19,LAGOS


,step,rows_before,rows_after,removed,note
0,"Raw extract: inproceedings, 2015-2025",1901774,1901774,0,
1,Keep conference/workshop keys,1901774,1898822,2952,prefixes kept: ['conf']


Step 2: missing values

Rules, and why:

- Drop a row with no author, no booktitle, no venue_key, or no year. Those fields are needed for every analysis in Tasks 2 to 6, and the leader’s guide says the same.
- Report but keep a row with a blank title. A paper with authors and a venue still counts toward popularity, so deleting it would be needless data loss. It matters only for the title-based duplicate check, and I’ll skip blank titles there.
- Ignore pages, ee, url and crossref. They aren’t used.

Why it prints two counts: a row can fail several rules, so the per-rule numbers can sum to more than the rows dropped. Report both in the methodology: the per-rule counts show which field is the problem, and the union is how many rows the step removed.

Two things to be aware of

This removes front matter only when it has no author. Earlier you saw rows like “Frontmatter” and “Preface” with empty author lists, and those go here. Front matter that lists the chairs as authors will stay in the data. I haven’t measured how many there are, so note it as a limitation, or check it with a title-keyword search later.
The author line also strips whitespace. That’s the beginning of author normalization, and it doesn’t touch the numeric suffixes.

In [30]:
def blank(s):
    """True where a value is missing or only whitespace."""
    return s.isna() | s.astype(str).str.strip().eq("")

# drop empty/whitespace-only author entries; strip stray spaces
df["authors"] = df["authors"].map(lambda L: [a.strip() for a in L if a and a.strip()])

drop_rules = {
    "no author":       df["authors"].str.len().eq(0),
    "blank booktitle": blank(df["booktitle"]),
    "blank venue_key": blank(df["venue_key"]),
    "blank year":      blank(df["year_raw"]),
}
flags = pd.concat(drop_rules, axis=1)

print("Rows failing each rule (rules can overlap):")
print(flags.sum(), "\n")
print("Rows failing at least one rule:", int(flags.any(axis=1).sum()))
print("Title blank (kept, reported only):", int(blank(df["title"]).sum()))

print("\nExamples of rows to drop:")
display(df.loc[flags.any(axis=1), ["paper_key", "title", "booktitle"]].head(10))

before = len(df)
df = df.loc[~flags.any(axis=1)].copy()
log_step("Drop rows missing author/booktitle/venue/year", before, len(df),
         "blank titles kept")
display(pd.DataFrame(funnel))

Rows failing each rule (rules can overlap):
no author          1613
blank booktitle       0
blank venue_key       0
blank year            0
dtype: int64 

Rows failing at least one rule: 1613
Title blank (kept, reported only): 0

Examples of rows to drop:


,paper_key,title,booktitle
2071,conf/dac/X22,Frontmatter.,DAC
23002,conf/clrimg/X22,"Color Imaging XXVII: Displaying, Processing, H...","Color Imaging: Displaying, Processing, Hardcop..."
32240,conf/sigir/X19,Preface: 4th Joint Workshop on BIRNDL at SIGIR...,BIRNDL@SIGIR
32378,conf/sigir/X20,BIRDS 2020 - Bridging the Gap between Informat...,BIRDS@SIGIR
33324,conf/sigir/X22,Preface to INRA 2022.,INRA/IWILDS@SIGIR
34227,conf/sigir/ZhouZHH16,Transfer Learning for Cross-Lingual Sentiment ...,SIGIR
38442,conf/transai/X23,Message from the Program Chairs TransAI 2023.,TransAI
38500,conf/transai/X23a,Message from the General Chairs TransAI 2023.,TransAI
40637,conf/csl/X17,"Front Matter, Table of Contents, Preface, Conf...",CSL
40674,conf/csl/X15,"Front Matter, Table of Contents, Preface, Conf...",CSL


,step,rows_before,rows_after,removed,note
0,"Raw extract: inproceedings, 2015-2025",1901774,1901774,0,
1,Keep conference/workshop keys,1901774,1898822,2952,prefixes kept: ['conf']
2,Drop rows missing author/booktitle/venue/year,1898822,1897209,1613,blank titles kept
